# Tuning extendido: LightGBM + target encoding + class_weight

Este notebook busca mejorar el mejor modelo de la baseline (LightGBM) en el escenario extendido. Todo se selecciona en **validación**; el test se usa una sola vez al final.

**Estrategia:**

1. `RandomizedSearchCV` con `StratifiedKFold(3)` sobre una submuestra estratificada de train (para que sea viable en local).
2. Comparación de cuatro variantes con los mejores hiperparámetros: One-Hot, One-Hot agrupando categorías raras (`min_frequency=0.01`), target encoding con validación cruzada interna y `class_weight='balanced'`.
3. Selección por `f1_macro` en validación; evaluación final en test.

**Target encoding:** `TargetEncoderCV` de `src/preprocessing_features.py` calcula las medias de clase *out-of-fold* en train. Así el modelo no ve el target de la fila que codifica y se evita la fuga típica del target encoding.

**Nota:** las clases ya están balanceadas (~25 % cada una), por lo que `class_weight` probablemente no ayude; se prueba para documentarlo.

In [1]:
import sys
sys.path.append('../..')

import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

from src.model_features import (
    GRID_LGBM_EXT,
    PARAMS_EXT,
    construir_lgbm_features,
    evaluar_modelo_ordinal,
    metricas_a_fila,
)
from src.preprocessing_features import FEATURES_EXT_PATH, preparar_dataset_features
from src.utils import (
    CLASES,
    REPORTS_DIR,
    SEED,
    cargar_dataset,
    fijar_semilla,
    guardar_modelo,
    split_estratificado,
)

sns.set_theme(style='whitegrid')
fijar_semilla(SEED)
print('Semilla:', SEED)

Semilla: 42


In [2]:
df = preparar_dataset_features(
    cargar_dataset(FEATURES_EXT_PATH),
    descartar_baseline=False,
)
X_train, X_val, X_test, y_train, y_val, y_test = split_estratificado(
    df,
    target='price_range',
    excluir=['price_clean', 'id'],
    seed=SEED,
)
print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)

# Submuestra estratificada solo para la búsqueda; el ajuste final usa todo train.
X_busqueda, _, y_busqueda, _ = train_test_split(
    X_train,
    y_train,
    train_size=min(25000, len(X_train)),
    stratify=y_train,
    random_state=SEED,
)
print('Submuestra de búsqueda:', X_busqueda.shape)

Train: (43131, 151) Val: (9243, 151) Test: (9243, 151)
Submuestra de búsqueda: (25000, 151)


In [3]:
base = construir_lgbm_features(
    X_busqueda,
    imputar=True,
    **PARAMS_EXT['lightgbm'],
)

busqueda = RandomizedSearchCV(
    estimator=base,
    param_distributions=GRID_LGBM_EXT,
    n_iter=12,
    scoring='f1_macro',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    random_state=SEED,
    n_jobs=1,
    verbose=2,
    refit=True,
)
busqueda.fit(X_busqueda, y_busqueda)

print('Mejor F1-Macro CV:', round(busqueda.best_score_, 4))
print('Mejores parámetros:', busqueda.best_params_)

resultados_cv = (
    pd.DataFrame(busqueda.cv_results_)
    .sort_values('rank_test_score')
    .reset_index(drop=True)
)
resultados_cv.to_csv(REPORTS_DIR / 'tuning_lightgbm_ext.csv', index=False)
resultados_cv.head(10)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END modelo__colsample_bytree=1.0, modelo__learning_rate=0.05, modelo__min_child_samples=40, modelo__n_estimators=700, modelo__num_leaves=63, modelo__reg_alpha=1.0, modelo__reg_lambda=0.0, modelo__subsample=1.0; total time=  21.7s
[CV] END modelo__colsample_bytree=1.0, modelo__learning_rate=0.05, modelo__min_child_samples=40, modelo__n_estimators=700, modelo__num_leaves=63, modelo__reg_alpha=1.0, modelo__reg_lambda=0.0, modelo__subsample=1.0; total time=  21.1s
[CV] END modelo__colsample_bytree=1.0, modelo__learning_rate=0.05, modelo__min_child_samples=40, modelo__n_estimators=700, modelo__num_leaves=63, modelo__reg_alpha=1.0, modelo__reg_lambda=0.0, modelo__subsample=1.0; total time=  22.2s
[CV] END modelo__colsample_bytree=0.6, modelo__learning_rate=0.03, modelo__min_child_samples=20, modelo__n_estimators=500, modelo__num_leaves=127, modelo__reg_alpha=1.0, modelo__reg_lambda=10.0, modelo__subsample=0.7; total time=  17.

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_modelo__subsample,param_modelo__reg_lambda,param_modelo__reg_alpha,param_modelo__num_leaves,param_modelo__n_estimators,param_modelo__min_child_samples,param_modelo__learning_rate,param_modelo__colsample_bytree,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
0,21.059999,1.531677,1.728666,0.135647,0.9,5.0,0.0,127,500,10,0.08,0.6,"{'modelo__subsample': 0.9, 'modelo__reg_lambda...",0.708092,0.706848,0.702520,0.705820,0.002388,1
1,15.754334,0.251949,1.717335,0.126349,0.7,10.0,1.0,127,500,20,0.03,0.6,"{'modelo__subsample': 0.7, 'modelo__reg_lambda...",0.707320,0.705562,0.703320,0.705401,0.001637,2
2,22.651669,0.365844,1.888332,0.099551,0.9,0.0,1.0,127,700,10,0.08,1.0,"{'modelo__subsample': 0.9, 'modelo__reg_lambda...",0.703374,0.708238,0.700574,0.704062,0.003166,3
3,8.574665,0.470357,0.711999,0.024712,0.9,10.0,0.0,31,500,40,0.08,0.6,"{'modelo__subsample': 0.9, 'modelo__reg_lambda...",0.705994,0.702544,0.701924,0.703487,0.001791,4
4,4.959999,0.094344,0.368334,0.006342,0.9,0.0,0.0,31,300,10,0.05,0.8,"{'modelo__subsample': 0.9, 'modelo__reg_lambda...",0.701522,0.707637,0.700946,0.703368,0.003027,5
5,14.661667,0.074460,0.755336,0.050391,1.0,0.0,0.0,255,200,40,0.08,0.8,"{'modelo__subsample': 1.0, 'modelo__reg_lambda...",0.700938,0.703063,0.701235,0.701745,0.000940,6
6,11.862668,0.247838,0.439332,0.002623,1.0,2.0,1.0,127,200,10,0.05,1.0,"{'modelo__subsample': 1.0, 'modelo__reg_lambda...",0.702816,0.704207,0.697993,0.701672,0.002663,7
7,13.493999,0.846397,0.666332,0.054431,0.8,5.0,1.0,255,200,10,0.03,0.8,"{'modelo__subsample': 0.8, 'modelo__reg_lambda...",0.698405,0.704119,0.702205,0.701576,0.002375,8
8,6.022659,0.172359,0.557669,0.153888,1.0,2.0,1.0,31,300,80,0.05,1.0,"{'modelo__subsample': 1.0, 'modelo__reg_lambda...",0.702552,0.703301,0.697976,0.701276,0.002354,9
9,20.177147,0.467135,1.569992,0.007068,1.0,0.0,1.0,63,700,40,0.05,1.0,"{'modelo__subsample': 1.0, 'modelo__reg_lambda...",0.700258,0.703032,0.700146,0.701145,0.001335,10


In [4]:
mejores = {
    clave.replace('modelo__', ''): valor
    for clave, valor in busqueda.best_params_.items()
}
print('Parámetros sin prefijo:', mejores)

modelo_busqueda = construir_lgbm_features(X_train, imputar=True, **mejores)
modelo_busqueda.fit(X_train, y_train)
met_busqueda = evaluar_modelo_ordinal(y_val, modelo_busqueda.predict(X_val))
print('Validación (mejor búsqueda):', {k: round(v, 4) for k, v in met_busqueda.items() if k != 'matriz_confusion'})

Parámetros sin prefijo: {'subsample': 0.9, 'reg_lambda': 5.0, 'reg_alpha': 0.0, 'num_leaves': 127, 'n_estimators': 500, 'min_child_samples': 10, 'learning_rate': 0.08, 'colsample_bytree': 0.6}
Validación (mejor búsqueda): {'accuracy': 0.7345, 'balanced_accuracy': 0.7342, 'f1_macro': 0.7341, 'f1_weighted': 0.7343, 'accuracy_adjacent': 0.9797, 'mae_ordinal': 0.2878, 'qwk': 0.8658}


In [5]:
variantes = []
configuraciones = [
    ('onehot', {}),
    (
        'target_encoding',
        {
            'target_encoding': [
                c for c in [
                    'neighbourhood_cleansed',
                    'property_type',
                    'neighbourhood',
                ]
                if c in X_train.columns
            ]
        },
    ),
    ('onehot_agrupado', {'min_frequency': 0.01}),
    ('class_weight', {'class_weight': 'balanced'}),
]

for nombre, extra in configuraciones:
    print('=' * 70)
    print('Variante:', nombre)
    modelo = construir_lgbm_features(
        X_train,
        imputar=True,
        **mejores,
        **extra,
    )
    modelo.fit(X_train, y_train)
    met = evaluar_modelo_ordinal(y_val, modelo.predict(X_val))
    print({k: round(v, 4) for k, v in met.items() if k != 'matriz_confusion'})
    variantes.append(
        metricas_a_fila(
            f'lightgbm_ext_{nombre}',
            met,
            escenario='extendido_tuning',
            **{k: str(v) for k, v in extra.items()},
        )
    )

variantes_df = pd.DataFrame(variantes).sort_values('f1_macro', ascending=False)
variantes_df.to_csv(REPORTS_DIR / 'tuning_variantes_ext.csv', index=False)
display(variantes_df)

Variante: onehot
{'accuracy': 0.7345, 'balanced_accuracy': 0.7342, 'f1_macro': 0.7341, 'f1_weighted': 0.7343, 'accuracy_adjacent': 0.9797, 'mae_ordinal': 0.2878, 'qwk': 0.8658}
Variante: target_encoding


ValueError: Some column names are not columns of the dataframe: {'neighbourhood'}

In [ ]:
mejor_variante = variantes_df.iloc[0]
nombre_variante = mejor_variante['modelo'].replace('lightgbm_ext_', '')
extra = dict(configuraciones[[c[0] for c in configuraciones].index(nombre_variante)][1])
print('Mejor variante:', nombre_variante)
print('Configuración extra:', extra)

modelo_final = construir_lgbm_features(
    X_train,
    imputar=True,
    **mejores,
    **extra,
)
modelo_final.fit(X_train, y_train)
met_test = evaluar_modelo_ordinal(y_test, modelo_final.predict(X_test))
print('Test:', {k: round(v, 4) for k, v in met_test.items() if k != 'matriz_confusion'})

pd.DataFrame(
    [
        metricas_a_fila(
            'lightgbm_ext_tuned',
            met_test,
            escenario='extendido_tuning',
            variante=nombre_variante,
            **mejores,
        )
    ]
).to_csv(REPORTS_DIR / 'lightgbm_ext_tuned_metricas.csv', index=False)

ConfusionMatrixDisplay(
    met_test['matriz_confusion'], display_labels=CLASES
).plot(cmap='Blues', values_format='d')
plt.title('LightGBM extendido + tuning - test')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    REPORTS_DIR / 'lightgbm_ext_tuned_matriz_confusion.png',
    dpi=150,
    bbox_inches='tight',
)
plt.show()

ruta = guardar_modelo(modelo_final, 'lightgbm_ext_tuned')
print('Modelo guardado en:', ruta)

config = {
    'modelo': 'lightgbm_ext_tuned',
    'variante': nombre_variante,
    'params': mejores,
    'extra': extra,
    'f1_macro_cv': float(busqueda.best_score_),
    'f1_macro_val': float(mejor_variante['f1_macro']),
    'accuracy_test': float(met_test['accuracy']),
    'f1_macro_test': float(met_test['f1_macro']),
    'accuracy_adjacent_test': float(met_test['accuracy_adjacent']),
}
with open(REPORTS_DIR / 'mejor_config_lightgbm_ext.json', 'w', encoding='utf-8') as archivo:
    json.dump(config, archivo, indent=2, ensure_ascii=False)
print(json.dumps(config, indent=2, ensure_ascii=False))

## Lectura de resultados

- `tuning_lightgbm_ext.csv` contiene todas las combinaciones evaluadas y su ranking.
- `tuning_variantes_ext.csv` compara One-Hot, target encoding y `class_weight`. Si el target encoding no mejora, se documenta y se conserva One-Hot: es más simple e interpretable.
- `lightgbm_ext_tuned_metricas.csv` es el resultado final del mejor esfuerzo para el 90 %.
- El test no participó en ninguna selección. La comparación con `lightgbm_ext` (configuración fija) y con `lightgbm_features` (geo) cuantifica la ganancia real del tuning.